In [2]:
! pip install spacy folium geopy feedparser pandas numpy
! python -m spacy download en_core_web_sm

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
/bin/bash: line 1: python: command not found


In [4]:
!pip install spacy folium geopy feedparser pandas numpy

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
!python -m spacy download en_core_web_sm

/bin/bash: line 1: python: command not found


In [3]:
import os
import re
import time
from datetime import datetime, timezone

import feedparser
import folium
import pandas as pd
import spacy
from folium.plugins import HeatMap, MarkerCluster
from geopy.geocoders import Nominatim

# -------------------------------------------------
# SETUP
# -------------------------------------------------
os.makedirs("archive", exist_ok=True)

# Make sure this model is installed once:
# python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
RSS_FEEDS = [
    "https://feeds.reuters.com/reuters/worldNews",
    "https://www.aljazeera.com/xml/rss/all.xml",
    "https://rss.nytimes.com/services/xml/rss/nyt/World.xml",
    "https://www.theguardian.com/world/middleeast/rss",
    "https://www.bbc.com/news/world/middle_east/rss.xml",
]

MAP_CENTER = [26.0, 50.0]
MAP_ZOOM = 5
MAP_FILE = "conflict_map.html"
INDEX_FILE = "index.html"

PRIMARY_TERMS = [
    "iran", "israel", "gaza", "lebanon", "tehran", "tel aviv", "jerusalem",
    "saudi arabia", "riyadh", "uae", "dubai", "abu dhabi", "qatar", "doha",
    "kuwait", "oman", "hormuz", "persian gulf", "red sea", "basra", "beirut",
    "damascus", "baghdad", "jeddah", "dammam", "fujairah", "muscat"
]

ACTION_TERMS = [
    "strike", "missile", "attack", "drone", "bomb", "rocket", "raid",
    "explosion", "blast", "refinery", "pipeline", "airport", "port",
    "power plant", "substation", "military base", "naval base", "airspace",
    "terminal", "grid", "telecom tower", "dam", "tunnel", "bunker",
    "government building", "ministry", "embassy", "parliament", "blackout",
    "runway", "airbase", "desalination", "reservoir", "satellite", "data center"
]

PLACE_HINTS = {
    # Iran
    "Tehran": "Tehran, Iran",
    "Natanz": "Natanz, Iran",
    "Fordow": "Fordow, Iran",
    "Bandar Abbas": "Bandar Abbas, Iran",
    "Bushehr": "Bushehr, Iran",
    "Kharg Island": "Kharg Island, Iran",
    "Assaluyeh": "Assaluyeh, Iran",
    "Isfahan": "Isfahan, Iran",
    "Shiraz": "Shiraz, Iran",
    "Zagros": "Zagros Mountains, Iran",

    # Israel / Palestine / Lebanon / Iraq / Syria
    "Tel Aviv": "Tel Aviv, Israel",
    "Jerusalem": "Jerusalem, Israel",
    "Ben Gurion": "Tel Aviv, Israel",
    "Gaza": "Gaza",
    "Beirut": "Beirut, Lebanon",
    "Lebanon": "Beirut, Lebanon",
    "Basra": "Basra, Iraq",
    "Baghdad": "Baghdad, Iraq",
    "Damascus": "Damascus, Syria",

    # Gulf
    "Dubai": "Dubai, United Arab Emirates",
    "Dubai Airport": "Dubai, United Arab Emirates",
    "Abu Dhabi": "Abu Dhabi, United Arab Emirates",
    "Fujairah": "Fujairah, United Arab Emirates",
    "Riyadh": "Riyadh, Saudi Arabia",
    "Jeddah": "Jeddah, Saudi Arabia",
    "Dammam": "Dammam, Saudi Arabia",
    "Abqaiq": "Abqaiq, Saudi Arabia",
    "Ras Tanura": "Ras Tanura, Saudi Arabia",
    "Doha": "Doha, Qatar",
    "Ras Laffan": "Ras Laffan Industrial City, Qatar",
    "Muscat": "Muscat, Oman",

    # Waterways / sea lanes
    "Hormuz": "Strait of Hormuz",
    "Strait of Hormuz": "Strait of Hormuz",
    "Persian Gulf": "Persian Gulf",
    "Gulf": "Persian Gulf",
    "Red Sea": "Red Sea",

    # Other
    "Washington": "Washington, DC, USA",
    "Pentagon": "Washington, DC, USA",
}

STRATEGIC_PLACE_PRIORITY = [
    "Ras Tanura", "Abqaiq", "Natanz", "Fordow", "Kharg Island", "Assaluyeh",
    "Bandar Abbas", "Bushehr", "Ras Laffan", "Dubai", "Abu Dhabi",
    "Tehran", "Tel Aviv", "Jerusalem", "Basra", "Beirut", "Doha", "Riyadh"
]

HIGH_SEVERITY = [
    "missile", "strike", "attack", "explosion", "bomb", "drone", "rocket", "blast"
]

INFRA_TERMS = [
    "refinery", "pipeline", "airport", "port", "power plant", "substation",
    "military base", "naval base", "dam", "telecom tower", "terminal",
    "airbase", "data center", "desalination"
]

# -------------------------------------------------
# HELPERS
# -------------------------------------------------
geolocator = Nominatim(user_agent="entropymap_ai_main")
geo_cache = {}


def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text or "")
    return re.sub(r"\s+", " ", text).strip()


def geocode_location(loc: str):
    if loc in geo_cache:
        return geo_cache[loc]
    try:
        point = geolocator.geocode(loc, timeout=10)
        if point:
            geo_cache[loc] = {"lat": point.latitude, "lon": point.longitude}
            return geo_cache[loc]
    except Exception:
        pass
    return None


def is_relevant(text: str) -> bool:
    t = text.lower()
    return any(p in t for p in PRIMARY_TERMS) or any(a in t for a in ACTION_TERMS)


def severity_score(text: str) -> int:
    score = 3
    t = text.lower()

    for w in HIGH_SEVERITY:
        if w in t:
            score += 4

    for w in INFRA_TERMS:
        if w in t:
            score += 3

    score += min(len(t.split()) // 12, 3)
    return min(score, 15)


def alert_level(score: int) -> str:
    if score >= 13:
        return "Critical"
    if score >= 10:
        return "High"
    if score >= 6:
        return "Medium"
    return "Low"


def alert_fill_color(level: str) -> str:
    return {
        "Critical": "#ff0033",
        "High": "#ff6600",
        "Medium": "#ffd400",
        "Low": "#33cc66",
    }.get(level, "#888888")


def detect_side(text: str) -> str:
    t = text.lower()
    if "iran" in t and ("israel" in t or "u.s." in t or "us " in t or "american" in t):
        return "Mixed"
    if "iran" in t:
        return "Iran"
    if "israel" in t:
        return "Israel"
    if "u.s." in t or "us " in t or "american" in t or "pentagon" in t or "washington" in t:
        return "US"
    return "Other"


def border_color_for_side(side: str) -> str:
    return {
        "Iran": "red",
        "Israel": "blue",
        "US": "green",
        "Mixed": "purple",
        "Other": "gray"
    }.get(side, "gray")


def detect_infrastructure(text: str) -> str:
    t = text.lower()

    if any(k in t for k in ["refinery", "pipeline", "oil", "gas", "terminal", "port", "lng"]):
        return "Energy / Maritime"
    if any(k in t for k in ["power plant", "grid", "substation", "blackout"]):
        return "Electric Grid"
    if any(k in t for k in ["airport", "airspace", "runway", "flight", "airbase"]):
        return "Aviation"
    if any(k in t for k in ["telecom", "tower", "satellite", "data center"]):
        return "Communication"
    if any(k in t for k in ["dam", "reservoir", "desalination", "water"]):
        return "Water"
    if any(k in t for k in ["military base", "naval base", "bunker", "missile base"]):
        return "Military"
    if any(k in t for k in ["government", "ministry", "embassy", "parliament"]):
        return "Government"
    return "Other"


def infra_icon(infra: str) -> str:
    return {
        "Energy / Maritime": "🛢",
        "Electric Grid": "⚡",
        "Aviation": "✈",
        "Communication": "📡",
        "Water": "💧",
        "Military": "🪖",
        "Government": "🏛",
        "Other": "•",
    }.get(infra, "•")


def deduplicate_items(items):
    seen = set()
    out = []
    for item in items:
        key = (
            item["title"].strip().lower(),
            round(item["lat"], 2),
            round(item["lon"], 2),
        )
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out


# -------------------------------------------------
# LOCATION EXTRACTION
# -------------------------------------------------
def extract_location_dictionary(text: str):
    matches = []
    t = text.lower()

    for hint, canon in PLACE_HINTS.items():
        if hint.lower() in t:
            matches.append((hint, canon))

    matches.sort(key=lambda x: len(x[0]), reverse=True)
    return matches[0][1] if matches else None


def extract_location_ai(text: str):
    doc = nlp(text)
    candidates = []

    for ent in doc.ents:
        if ent.label_ in {"GPE", "LOC", "FAC"}:
            ent_text = ent.text.strip()
            if ent_text:
                candidates.append(ent_text)

    for strategic in STRATEGIC_PLACE_PRIORITY:
        for cand in candidates:
            if strategic.lower() in cand.lower() or cand.lower() in strategic.lower():
                return PLACE_HINTS.get(strategic, strategic)

    for cand in candidates:
        for hint, canon in PLACE_HINTS.items():
            if hint.lower() == cand.lower() or hint.lower() in cand.lower() or cand.lower() in hint.lower():
                return canon

    if candidates:
        return candidates[0]

    return None


def extract_location_hybrid(text: str):
    loc_ai = extract_location_ai(text)
    if loc_ai:
        return loc_ai, "ai"

    loc_dict = extract_location_dictionary(text)
    if loc_dict:
        return loc_dict, "dictionary"

    return None, None


def confidence_score(text: str, loc_source: str, infrastructure: str) -> float:
    score = 0.35
    t = text.lower()

    if loc_source == "ai":
        score += 0.30
    elif loc_source == "dictionary":
        score += 0.20

    if any(k in t for k in PRIMARY_TERMS):
        score += 0.15
    if any(k in t for k in ACTION_TERMS):
        score += 0.15
    if infrastructure != "Other":
        score += 0.10

    return round(min(score, 0.99), 2)


def ai_summary(title: str, location: str, infra: str, level: str) -> str:
    return f"{level} alert involving {infra.lower()} near {location}."


# -------------------------------------------------
# PARSING
# -------------------------------------------------
def parse_rss():
    incidents = []
    now_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

    for url in RSS_FEEDS:
        try:
            feed = feedparser.parse(url)
        except Exception:
            continue

        source_title = clean_text(feed.get("feed", {}).get("title", url))
        entries = getattr(feed, "entries", [])[:20]

        for entry in entries:
            title = clean_text(getattr(entry, "title", ""))
            summary = clean_text(getattr(entry, "summary", ""))
            text = f"{title} {summary}"

            if not is_relevant(text):
                continue

            infra = detect_infrastructure(text)
            location, loc_source = extract_location_hybrid(text)

            if not location:
                continue

            geo = geocode_location(location)
            time.sleep(1)

            if not geo:
                continue

            sev = severity_score(text)
            level = alert_level(sev)
            side = detect_side(text)
            conf = confidence_score(text, loc_source, infra)
            summary_line = ai_summary(title, location, infra, level)

            incidents.append({
                "published": now_utc,
                "title": title,
                "location": location,
                "location_source": loc_source,
                "lat": geo["lat"],
                "lon": geo["lon"],
                "severity": sev,
                "alert_level": level,
                "side": side,
                "infrastructure": infra,
                "confidence": conf,
                "ai_summary": summary_line,
                "source": source_title,
                "link": getattr(entry, "link", ""),
            })

    return deduplicate_items(incidents)


# -------------------------------------------------
# MAP
# -------------------------------------------------
def build_map(incidents):
    m = folium.Map(
        location=MAP_CENTER,
        zoom_start=MAP_ZOOM,
        tiles="CartoDB dark_matter"
    )

    cluster = MarkerCluster().add_to(m)

    for i in incidents:
        folium.CircleMarker(
            location=[i["lat"], i["lon"]],
            radius=max(6, i["severity"]),
            color=border_color_for_side(i["side"]),
            weight=2,
            fill=True,
            fill_color=alert_fill_color(i["alert_level"]),
            fill_opacity=0.75,
            popup=(
                f"<b>{i['title']}</b><br>"
                f"<b>AI summary:</b> {i['ai_summary']}<br>"
                f"<b>Location:</b> {i['location']}<br>"
                f"<b>Location source:</b> {i['location_source']}<br>"
                f"<b>Side:</b> {i['side']}<br>"
                f"<b>Infrastructure:</b> {infra_icon(i['infrastructure'])} {i['infrastructure']}<br>"
                f"<b>Alert:</b> {i['alert_level']}<br>"
                f"<b>Severity:</b> {i['severity']}<br>"
                f"<b>Confidence:</b> {i['confidence']}<br>"
                f"<b>Source:</b> {i['source']}<br>"
                f"<a href='{i['link']}' target='_blank'>Open article</a>"
            ),
            tooltip=f"{infra_icon(i['infrastructure'])} {i['location']} | {i['alert_level']} | conf {i['confidence']}"
        ).add_to(cluster)

        if i["severity"] >= 10:
            folium.Circle(
                location=[i["lat"], i["lon"]],
                radius=200000,
                color="#ff3333",
                weight=1,
                fill=False,
                opacity=0.35,
                popup="~200 km impact zone"
            ).add_to(m)

    if incidents:
        HeatMap(
            [[x["lat"], x["lon"], x["severity"]] for x in incidents],
            radius=35,
            blur=20
        ).add_to(m)

    return m


# -------------------------------------------------
# DASHBOARD
# -------------------------------------------------
def make_dashboard(df: pd.DataFrame):
    style_block = """
    <style>
    body {
        background: #0f1115;
        color: #e6e6e6;
        font-family: Arial, sans-serif;
        margin: 20px;
    }
    h1, h2, h3 {
        color: #ffffff;
    }
    .note {
        color: #b8c0cc;
        margin-bottom: 18px;
    }
    iframe {
        width: 100%;
        height: 700px;
        border: 1px solid #333;
        background: #111;
    }
    table {
        border-collapse: collapse;
        width: 100%;
        font-size: 13px;
        background: #171a21;
    }
    th, td {
        border: 1px solid #2a2f3a;
        padding: 8px;
        vertical-align: top;
    }
    th {
        background: #202531;
        color: #fff;
    }
    tr:nth-child(even) {
        background: #131720;
    }
    a {
        color: #6cb6ff;
    }
    .statbar {
        display: flex;
        gap: 14px;
        margin: 14px 0 20px 0;
        flex-wrap: wrap;
    }
    .card {
        min-width: 140px;
        padding: 12px 14px;
        border-radius: 10px;
        background: #1a1f29;
        border: 1px solid #2e3440;
        box-shadow: 0 0 10px rgba(0,0,0,0.25);
    }
    .card .label {
        font-size: 12px;
        color: #aab4c3;
    }
    .card .value {
        font-size: 24px;
        font-weight: bold;
        margin-top: 4px;
    }
    .legend {
        margin-top: 10px;
        padding: 12px;
        background: #171a21;
        border: 1px solid #2a2f3a;
        border-radius: 10px;
    }
    .badge {
        display: inline-block;
        padding: 3px 8px;
        border-radius: 999px;
        font-size: 12px;
        font-weight: bold;
    }
    .critical { background: #ff0033; color: white; }
    .high     { background: #ff6600; color: white; }
    .medium   { background: #ffd400; color: black; }
    .low      { background: #33cc66; color: black; }
    </style>
    """

    table_html = df.to_html(index=False, escape=False)
    last_update = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")

    total_incidents = len(df)
    critical_count = int((df["alert_level"] == "Critical").sum()) if not df.empty else 0
    high_count = int((df["alert_level"] == "High").sum()) if not df.empty else 0
    energy_count = int((df["infrastructure"] == "Energy / Maritime").sum()) if not df.empty else 0
    aviation_count = int((df["infrastructure"] == "Aviation").sum()) if not df.empty else 0

    stats_html = f"""
    <div class="statbar">
      <div class="card"><div class="label">Total Incidents</div><div class="value">{total_incidents}</div></div>
      <div class="card"><div class="label">Critical</div><div class="value">{critical_count}</div></div>
      <div class="card"><div class="label">High</div><div class="value">{high_count}</div></div>
      <div class="card"><div class="label">Energy</div><div class="value">{energy_count}</div></div>
      <div class="card"><div class="label">Aviation</div><div class="value">{aviation_count}</div></div>
    </div>
    """

    legend_html = """
    <div class="legend">
      <b>Alert legend</b><br><br>
      <span class="badge critical">Critical</span>
      <span class="badge high">High</span>
      <span class="badge medium">Medium</span>
      <span class="badge low">Low</span>
      <br><br>
      <b>Border colors:</b><br>
      Red = Iran | Blue = Israel | Green = US | Purple = Mixed | Gray = Other
    </div>
    """

    html = f"""<html>
<head>
<meta charset="utf-8">
<title>EntropyMap AI Dashboard</title>
{style_block}
</head>
<body>
<h1>EntropyMap AI Dashboard</h1>
<div class="note">Last update: {last_update} | Dark theme enabled | AI-enhanced location extraction</div>
{stats_html}
{legend_html}
<br>
<iframe src="{MAP_FILE}"></iframe>
<h2>Detected incidents</h2>
{table_html}
</body>
</html>
"""

    with open(INDEX_FILE, "w", encoding="utf-8") as f:
        f.write(html)


# -------------------------------------------------
# MAIN
# -------------------------------------------------
def main():
    incidents = parse_rss()

    df = pd.DataFrame(incidents, columns=[
        "published", "location", "location_source", "side", "infrastructure",
        "alert_level", "severity", "confidence", "ai_summary", "source", "title", "link"
    ])

    # live files
    build_map(incidents).save(MAP_FILE)
    make_dashboard(df)

    # latest machine-readable files
    df.to_json("incidents_latest.json", orient="records", force_ascii=False, indent=2)
    df.to_csv("incidents_latest.csv", index=False)

    # archive snapshot
    now_str = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
    df.to_json(
        f"archive/incidents_{now_str}.json",
        orient="records",
        force_ascii=False,
        indent=2
    )

    print(f"Incidents detected: {len(incidents)}")
    print(f"Wrote {MAP_FILE}, {INDEX_FILE}, incidents_latest.json, incidents_latest.csv")


if __name__ == "__main__":
    main()

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.